# 📖 Theory Notebook — Tutorial 2: EEG Preprocessing
### 4th Year · Biomedical Signals & Applications

---

> **How to use this notebook:**
> Open it side-by-side with `Tutorial_2_EEG_Preprocessing.ipynb`.
> Before running each code step, read the matching theory section here.
> No maths background required — every concept is explained with a real-world analogy first.

---

## What is this tutorial about?

When we record EEG from a person's scalp, the signal we get is **messy**.
It contains the brain activity we care about, but also:

- 🙈 Eye blinks and eye movements
- 💪 Jaw and neck muscle tension
- ❤️ The person's heartbeat
- ⚡ Electrical interference from the power grid in the walls
- 🔌 Noise from loose or imperfect electrode contacts

**Preprocessing** is the process of cleaning up all of that noise so we are left with just the brain signal — ready for analysis.

Think of it like washing and preparing vegetables before cooking. You wouldn't skip that step even if it takes time, because the quality of the final dish depends on it.

---

## The Preprocessing Pipeline at a Glance

Each step in Tutorial 2 solves one specific problem:

| Step | What we do | Problem it solves |
|------|-----------|-------------------|
| 0 | Load the raw data | Get the recording into Python |
| 1 | Re-reference | Remove bias from the original reference electrode |
| 2 | Filter (notch + bandpass) | Remove power-line hum and out-of-range noise |
| 3 | Find and fix bad channels | Recover broken electrodes |
| 4 | ICA | Remove eye blinks, muscle, heartbeat |
| 5 | Epoch | Cut the recording into individual trials |
| 6 | Reject bad epochs | Discard any trials that are still too noisy |
| 7 | Save | Store clean data for Tutorial 3 |

---

---
## 1. What Does Raw EEG Look Like?

Before we clean anything, let's understand what we're working with.

---

### The scale problem

EEG signals are incredibly small. We measure them in **microvolts (µV)** — one millionth of a volt.

| Signal | Typical size |
|--------|-------------|
| AA battery | 1,500,000 µV |
| Eye blink artefact | 200 – 1000 µV |
| Jaw clench (muscle) | 100 – 1000 µV |
| **Brain EEG signal** | **5 – 50 µV** |

An eye blink can be **20 times larger** than the brain signal we care about.
This is why cleaning the data is not optional — without it, our analysis sees mostly noise.

---

### Where does the noise come from?

There are three sources of contamination in raw EEG:

**1. Your own body (physiological artefacts)**
Your eyes, jaw muscles, and heart all produce electrical signals.
These are picked up by the same electrodes that record your brain.
The tricky part: they overlap in frequency with brain signals, so you can't simply filter them out.

**2. The power grid (electrical interference)**
The electrical wiring in the walls radiates an electromagnetic field at exactly 50 Hz (in Europe) or 60 Hz (in the USA).
EEG cables act like antennas and pick this up as a constant sinusoidal hum.

**3. The equipment itself**
Electrode gel dries out. Cables develop micro-breaks. Sweat changes the conductivity.
All of these produce noisy or completely dead channels.

---

> 💡 **Key idea:** The brain signal is real and it is there — it's just hidden inside a lot of noise. Preprocessing reveals it; it doesn't create it.

---

---
## 2. Re-Referencing — Choosing a Neutral Baseline

---

### Why does a reference electrode exist?

An EEG amplifier cannot measure voltage in absolute terms.
It can only measure the **difference** between two points:
the active electrode (on the scalp) and a reference electrode (somewhere else).

Every channel you see is actually:

```
What you measure at channel X  =  Brain signal at X  −  Signal at the reference
```

This means the reference electrode contaminates every single channel.
If the reference electrode is near some brain activity, that activity gets subtracted from all 64 channels at once.

---

> 🟢 **Analogy:** Imagine measuring the height of every building in a city. You need a baseline — sea level. If instead you measured heights relative to the roof of one tall building in the city, every measurement would be shifted. Some buildings would appear taller, others shorter. Changing to sea level (a neutral baseline) gives you the true heights.

---

### The average reference

The most common solution is the **average reference**:

1. At every moment in time, compute the average voltage across all 64 electrodes
2. Subtract this average from each electrode

```
Clean signal at channel X  =  Recorded signal at X  −  Average of ALL channels
```

The average of many electrodes spread across the whole scalp is approximately neutral — it doesn't favour any brain region. This makes it a much better reference than any single electrode.

---

### ⚠️ Important order rule

You must fix bad channels **before** computing the average reference.

Why? If one electrode is broken and recording pure noise, it inflates the average.
That noisy average then gets subtracted from every good channel — spreading the noise everywhere.

**Fix bad channels first → then compute the clean average → then re-reference.**

---

---
## 3. Filtering — Keeping Only the Frequencies We Need

---

### What is frequency in EEG?

The brain produces rhythmic oscillations — the voltage goes up and down repeatedly.
**Frequency** describes how fast those oscillations are, measured in Hz (cycles per second).

A 10 Hz signal completes 10 full up-down-up cycles in one second.
A 1 Hz signal completes only 1 cycle per second — much slower.

Different types of brain activity produce oscillations at different frequencies.
Noise also lives at specific frequencies. This is why filtering is so powerful.

---

### EEG frequency bands — what lives where

| Frequency | What lives there | Keep or remove? |
|-----------|-----------------|-----------------|
| < 1 Hz | Electrode drift, sweat, slow breathing | ❌ Remove |
| 1 – 4 Hz (Delta) | Deep sleep | ✅ Keep |
| 4 – 8 Hz (Theta) | Memory, drowsiness | ✅ Keep |
| 8 – 13 Hz (Alpha/Mu) | Relaxation, motor idle rhythm | ✅ Keep — crucial! |
| 13 – 30 Hz (Beta) | Active thinking, motor imagery | ✅ Keep — crucial! |
| 30 – 40 Hz (Gamma) | High cognition (edge of usefulness) | ✅ Keep with caution |
| **50 Hz (Europe) / 60 Hz (USA)** | **Power-line noise** | **❌ Remove** |
| > 40 Hz | Mostly muscle noise | ❌ Remove |

---

> 🟢 **Analogy:** A filter is like sunglasses. They block ultraviolet light (which you don't want) while letting visible light through (which you do want). A bandpass filter blocks everything below 1 Hz and above 40 Hz, and lets everything in between pass through unchanged.

---

### Two types of filter

**Notch filter** — removes one very narrow frequency band  
Used to delete the 50/60 Hz power-line spike. It's like a surgical scalpel — very precise, leaves everything else untouched.

**Bandpass filter** — keeps only a defined range  
We keep 1–40 Hz and remove everything outside. This handles both the slow drifts (< 1 Hz) and the high-frequency muscle noise (> 40 Hz) in a single step.

---

### Zero-phase filtering — why timing is preserved

A naive filter introduces a time shift — the filtered signal appears slightly earlier or later than reality.

For EEG analysis, timing is everything. If the brain responds at 300 ms after a stimulus, we need that peak to appear at exactly 300 ms — not at 280 ms due to a filter artefact.

MNE uses **zero-phase filtering** by default: it applies the filter twice (once forward, once backward), and the two time shifts cancel exactly. You don't need to do anything special — MNE handles this automatically.

---

> 💡 **European vs USA data:**
> Europe (including Greece): use `notch_filter(freqs=50)`
> USA: use `notch_filter(freqs=60)`
> When unsure: plot the power spectrum first — look for a sharp spike!

---

---
## 4. Bad Channels — Finding and Fixing Broken Electrodes

---

### What makes a channel "bad"?

In every real EEG recording, some electrodes don't work perfectly. Common reasons:

- **Poor contact:** Not enough gel, or the gel dried out → the signal is very small or flat
- **Broken cable:** Damaged wire causes dropout or pure noise
- **High skin impedance:** Skin was not prepared properly → too much resistance → noise dominates
- **Bridging:** Too much gel connects two adjacent electrodes → they record identical signals

---

> 🟢 **Analogy:** Bad channels are like a faulty microphone in a recording studio. One microphone is buzzing or silent, but the other microphones are fine. You don't throw away the whole recording — you fix that one microphone. In EEG, we "fix" it by estimating what it should have recorded based on what the surrounding good channels recorded.

---

### How to spot a bad channel

When you look at the raw EEG traces, a bad channel stands out:

- **Flat line** — records nothing; the trace sits at zero
- **Extreme noise** — fluctuates wildly, 5–10× larger than its neighbours
- **Constant offset** — same shape as neighbours but at a very different level

---

### Spherical spline interpolation — reconstructing the signal

Once we mark a channel as bad, MNE can reconstruct it using the surrounding good channels.

The method works in three intuitive steps:
1. Imagine all the good electrode readings as data points on a smooth sphere (the head)
2. Fit a smooth surface through all those known points
3. Read off the estimated value at the location of the bad electrode

The result is not identical to what the real electrode would have recorded, but it is a reasonable estimate — good enough to prevent the bad channel from corrupting the rest of the analysis.

---

> ⚠️ **Rules of thumb:**
> - It's safe to interpolate up to about 5% of channels (~3 out of 64)
> - If more than 15–20% of channels are bad, the recording is probably too poor to use
> - Always interpolate before applying the average reference

---

---
## 5. ICA — Separating Brain from Body Signals

This is the most important and most powerful step in EEG preprocessing. Take a moment to really understand it.

---

### The problem

Eye blinks produce signals up to 1000 µV — far larger than any brain signal.
The problem is that eye blinks produce voltage in the **same frequency range** as brain activity (1–10 Hz).

A bandpass filter cannot separate them — they overlap in frequency.
We need a completely different approach.

---

### The idea behind ICA

**ICA** stands for Independent Component Analysis.

Here is the core idea: the EEG signal recorded at each electrode is a **mixture** of signals from many different sources. Some sources are brain regions. Others are eye muscles, jaw muscles, and the heart. Each electrode picks up a different blend of all these sources.

ICA is a mathematical method that **unmixes** these blended signals and recovers the original sources. It can do this because brain signals and physiological artefacts are **statistically independent** — knowing when someone blinks tells you absolutely nothing about what their alpha rhythm is doing at that moment. ICA exploits this independence to disentangle the sources.

---

> 🟢 **Analogy — the cocktail party problem:**
> Imagine a room with three people talking simultaneously, and three microphones placed at different positions. Each microphone records a different mixture of all three voices.
>
> ICA is the mathematical equivalent of a person who listens to all three microphones and figures out what each individual person was saying — purely by detecting that the three voices are statistically independent of each other.
>
> In EEG: the microphones are the electrodes, the voices are the different sources (brain areas, eye muscles, heart), and ICA is the unmixing algorithm.

---

### What ICA gives you

After running ICA on 64-channel EEG, you get 64 **independent components (ICs)**.
Each component has two things to look at:

- **A scalp topography** — a colour map showing where this source projects onto the scalp
- **A time series** — the waveform of that source's activity over the whole recording

You then look at these and decide: is this a brain signal (keep it) or an artefact (remove it)?

---

### How to identify artefacts

| Component type | What the scalp map looks like | What the time series looks like |
|----------------|------------------------------|--------------------------------|
| **Eye blink** | Large blob at the very front of the head | Regular sharp triangular spikes — one per blink |
| **Lateral eye movement** | Asymmetric: positive left-front, negative right-front | Step-like jumps when eyes move sideways |
| **Heartbeat** | Spread across scalp, stronger near temples | Regular pulse shape every ~1 second |
| **Jaw muscle** | Spots at the sides/bottom of the scalp map | Messy bursts of high-frequency noise |
| **Brain — alpha** | Smooth blob at the back of the head | Waxing and waning ~10 Hz rhythm |
| **Brain — motor mu** | Two symmetric spots over C3 and C4 | 8–12 Hz oscillation that dips during movement imagery |

---

### The golden rule of ICA

> ⚠️ **When in doubt — keep the component.**
>
> Removing a brain component **permanently destroys** neural information.
> Keeping a mild artefact component is far less damaging than throwing away real brain signal.
>
> In a typical session: remove **1–3 components** (usually eye blinks and possibly a heartbeat).
> If you feel tempted to remove more than 5, stop and reconsider.

---

### Why we filter BEFORE running ICA

ICA needs clean-enough data to find the independent components correctly.

Very slow drifts (below 1 Hz) are much larger than anything else in the signal.
If those drifts are present, they dominate the analysis and ICA cannot find the subtler independence of brain vs eye vs muscle sources.

The 1 Hz high-pass filter we applied in Step 2 specifically solves this.

Also: ICA works on **long, continuous data** — not on short segments.
This is why we run ICA on the full raw recording, before cutting it into epochs.

---

---
## 6. Epoching — Cutting the Recording into Trials

---

### From one long recording to many short windows

After cleaning the continuous EEG, we need to connect the brain signal to the experimental events — the specific moments when a cue was shown and the subject began imagining a movement.

**Epoching** does this: it cuts the recording into fixed-length windows, each one centred on an event marker.

---

> 🟢 **Analogy:** You have filmed a one-hour tennis match. You want to study the moment of each serve. Epoching is like finding every serve in the video and cutting out a 5-second clip centred on that moment. You end up with 80 short clips, all aligned to the same event.

---

### Defining an epoch

Each epoch needs three things:

- **The event marker** — the time stamp of the event we care about (T1 = left hand cue, T2 = right hand cue)
- **tmin** — how much time to include *before* the event (we use −0.5 s as a baseline window)
- **tmax** — how much time to include *after* the event (we use +3.0 s to cover the full imagery period)

So each epoch spans from 0.5 seconds before the cue to 3 seconds after — a total window of 3.5 seconds.

---

### Baseline correction

After cutting epochs, we apply **baseline correction**:

1. Compute the average voltage in each channel during the pre-stimulus period (−0.5 to 0 s)
2. Subtract that average from the whole epoch

This zeroes out any slow drift that happened to be present at the moment the cue appeared.
Without this, trials with different DC offsets at their starting point would not be directly comparable.

---

> 🟢 **Analogy:** Measuring how much a runner improves during a 100 m sprint. You don't care where they started on the track — you zero their starting position and only measure the change from start to finish. Baseline correction does this for each EEG trial.

---

### Epoch rejection

Even after ICA, some trials will still contain extreme noise — a sudden jaw clench, a large head movement. We discard those trials automatically using a threshold:

> If any channel in any trial exceeds **±150 µV** at any point, that trial is thrown away.

- **Too strict (e.g. 50 µV):** You reject too many trials → not enough data for reliable analysis
- **Too lenient (e.g. 500 µV):** Dirty trials survive → they corrupt your analysis
- **Typical (100–150 µV):** Good balance for most adult subjects

Aim to keep at least 70% of your trials. If you're losing more, something in the earlier steps needs adjustment.

---

---
## 7. Why the Order Matters

The preprocessing steps must happen in a specific order. Each step assumes the previous one has already been done correctly.

---

| Step | Must come before... | Why |
|------|-------------------|-----|
| Fix bad channels | Average reference | A broken channel corrupts the average — fix first |
| Filter (1 Hz+) | ICA | Slow drifts confuse ICA — remove them first |
| ICA | Epoching | ICA needs long continuous data — epochs break the signal |
| ICA | Rejection | ICA removes the big artefacts so rejection threshold can be reasonable |

---

> ⚠️ **The most common mistake:** Epoching the data and then trying to run ICA on the short segments. This doesn't work well because ICA needs many seconds of data to estimate statistical independence reliably. Always run ICA on the full continuous recording, then epoch.

---

## Summary — The Whole Pipeline in One Picture

```
Raw EEG (messy, continuous)
       │
       ▼
  [1] Re-reference to average  →  removes reference bias
       │
       ▼
  [2] Notch filter (50/60 Hz)  →  removes power-line hum
  [2] Bandpass filter (1–40 Hz) → removes drifts and muscle noise
       │
       ▼
  [3] Mark bad channels         →  identify broken electrodes
  [3] Interpolate bad channels  →  reconstruct from neighbours
       │
       ▼
  [4] Fit ICA                   →  learn the independent sources
  [4] Identify artefact ICs     →  find eye/muscle/heartbeat components
  [4] Apply ICA                 →  remove artefacts, reconstruct EEG
       │
       ▼
  [5] Extract event markers     →  find T1/T2 timestamps
  [5] Cut epochs (−0.5 to +3 s) → create individual trials
  [5] Baseline correct          →  zero the pre-stimulus mean
       │
       ▼
  [6] Reject bad epochs         →  discard trials above ±150 µV
       │
       ▼
  Clean epochs — ready for Tutorial 3 🧠
```

---

> 💡 **The most important thing to remember:**
> You are a detective, not a sculptor.
> Your job is to **reveal** the brain signal that is already there — not to **shape** the data until it looks like what you expect.
> Every decision should be guided by: "Is this artefact, or is this brain activity?"
> If you are not sure — keep it.

---

---
## 8. Quick Glossary

| Term | Plain-English meaning |
|------|-----------------------|
| **Microvolt (µV)** | One millionth of a volt — the unit EEG is measured in |
| **Artefact** | Any signal in the EEG that did NOT come from the brain |
| **Reference electrode** | The electrode everything else is measured against |
| **Average reference** | Using the mean of all electrodes as the reference — more neutral than any single electrode |
| **Notch filter** | Removes one very narrow frequency range (e.g. exactly 50 Hz) |
| **Bandpass filter** | Keeps only a defined frequency range (e.g. 1–40 Hz) |
| **High-pass filter** | Removes very slow signals (below a cutoff, e.g. 1 Hz) |
| **Low-pass filter** | Removes very fast signals (above a cutoff, e.g. 40 Hz) |
| **Interpolation** | Reconstructing a bad channel from its neighbours |
| **ICA** | Independent Component Analysis — unmixes the EEG into separate brain and artefact sources |
| **Independent component (IC)** | One of the separated sources found by ICA |
| **Epoch** | A short fixed-length segment of EEG aligned to one event |
| **Baseline correction** | Subtracting the pre-stimulus mean from each trial |
| **Epoch rejection** | Discarding trials with extreme amplitude |
| **T1 / T2** | Event markers in this dataset: T1 = left hand cue, T2 = right hand cue |
| **EOG** | Electrooculogram — electrical signal from eye movements |
| **EMG** | Electromyogram — electrical signal from muscles |

---

> ✅ **You are now ready to run Tutorial 2.**
> Keep this notebook open alongside the code notebook.
> When you reach each step in the code, come back here to remind yourself what it does and why.